In [171]:
import pandas as pd
import polars as pl
from datetime import datetime, timedelta,date
import numpy as np
from pathlib import Path
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score,roc_curve, auc,mean_squared_error,
# precision_recall_curve,log_loss,roc_auc_score,average_precision_score,f1_score)
# from sklearn.model_selection import train_test_split,RepeatedKFold,cross_val_score,TimeSeriesSplit,GridSearchCV
# from tqdm import tqdm
# from sklearn.preprocessing import StandardScaler,MinMaxScaler,RobustScaler,OneHotEncoder
# from sklearn.pipeline import Pipeline
# from sklearn.neural_network import MLPRegressor,MLPClassifier
# from sklearn.compose import ColumnTransformer
# from sklearn.impute import SimpleImputer
# from sklearn.base import clone
# from sklearn.calibration import calibration_curve

In [172]:
data = pd.DataFrame() 
years = ["2022","2023","2024","2025"]

for y in years:
    PATH = f"../data/processed/statcast_{y}.parquet"
    data = pd.concat([data, pd.read_parquet(PATH)])

In [173]:
data = pl.DataFrame(data)

In [174]:
data.columns

['pitch_type',
 'game_date',
 'release_speed',
 'release_pos_x',
 'release_pos_z',
 'player_name',
 'batter',
 'pitcher',
 'events',
 'description',
 'spin_dir',
 'spin_rate_deprecated',
 'break_angle_deprecated',
 'break_length_deprecated',
 'zone',
 'des',
 'game_type',
 'stand',
 'p_throws',
 'home_team',
 'away_team',
 'type',
 'hit_location',
 'bb_type',
 'balls',
 'strikes',
 'game_year',
 'pfx_x',
 'pfx_z',
 'plate_x',
 'plate_z',
 'on_3b',
 'on_2b',
 'on_1b',
 'outs_when_up',
 'inning',
 'inning_topbot',
 'hc_x',
 'hc_y',
 'tfs_deprecated',
 'tfs_zulu_deprecated',
 'umpire',
 'sv_id',
 'vx0',
 'vy0',
 'vz0',
 'ax',
 'ay',
 'az',
 'sz_top',
 'sz_bot',
 'hit_distance_sc',
 'launch_speed',
 'launch_angle',
 'effective_speed',
 'release_spin_rate',
 'release_extension',
 'game_pk',
 'fielder_2',
 'fielder_3',
 'fielder_4',
 'fielder_5',
 'fielder_6',
 'fielder_7',
 'fielder_8',
 'fielder_9',
 'release_pos_y',
 'estimated_ba_using_speedangle',
 'estimated_woba_using_speedangle',
 'w

In [175]:
#missed balls
data = (
    data
    .with_columns (
        (
            (
                (pl.col("plate_x") < -0.7803)
                | (pl.col("plate_x") > 0.7803)

                | (pl.col("plate_z") < pl.col("sz_bot"))
                | (pl.col("plate_z") > pl.col("sz_top"))
            )
           & (pl.col("description")=="called_strike")
        )
        
         .cast(pl.Int8)
        .alias("is_missed_ball")
    )
)

In [176]:
#missed strikes
data = (
    data
    .with_columns (
        (
            (
                (pl.col("plate_x") >= -0.7803)
                & (pl.col("plate_x") <= 0.7803)

                & (pl.col("plate_z") >= pl.col("sz_bot"))
                & (pl.col("plate_z") <= pl.col("sz_top"))
            )
           & (pl.col("description")=="ball")
        )
        
         .cast(pl.Int8)
        .alias("is_missed_strike")
    )
)

In [177]:
# source:
#https://tangotiger.com/index.php/site/article/statcast-lab-swing-take-and-a-primer-on-run-value

def add_shadow_zones(df,plate_in = 8.5, ball_in =2.9, waste_in =6.0, name = "attack_zone"):
    """
    Heart  = inside the rule-book zone
    Shadow = within one ball-width of the zone edge
    Chase  = within `ball_in + waste_in` of the zone edge
    Waste  = beyond that
    """
    required = ["plate_x", "plate_z", "sz_top", "sz_bot"]
    missing = pl.any_horizontal(*[pl.col(c).is_null() for c in required])

    # df = df.lazy()
    # output
    output = (
        df.with_columns(
            sz_mid = (pl.col("sz_top") + pl.col("sz_bot")) / 2,
            sz_half = ((pl.col("sz_top") - pl.col("sz_bot")) / 2) * 12,
        )
        .with_columns(
            dx = pl.col("plate_x").abs() * 12 - plate_in,
            dz = (pl.col("plate_z") - pl.col("sz_mid")).abs() * 12 - pl.col("sz_half"),
        )
        .with_columns(
            dist = pl.max_horizontal(pl.col("dx"), pl.col("dz"))
        )
         .with_columns(
            pl.when(missing)
              .then(pl.lit(None, dtype=pl.Utf8))
              .when(pl.col("dist") <= -ball_in).then(pl.lit("Heart"))
              .when(pl.col("dist") <= 0).then(pl.lit("Shadow"))
             .when(pl.col("dist") <= ball_in).then(pl.lit("Shadow"))
              .when(pl.col("dist") <= ball_in + waste_in).then(pl.lit("Chase"))
              .otherwise(pl.lit("Waste"))
              .alias(name)
        )
        .drop(["sz_mid","sz_half","dx","dz","dist"])
    )
    return output
data = add_shadow_zones(data)

In [178]:
data = data.filter(pl.col("description").is_in(["called_strike", "ball"]) & 
                (pl.col("plate_x").is_not_null()) &
                (pl.col("sz_top").is_not_null())
                )
data = data.with_columns((pl.col("is_missed_ball") | pl.col("is_missed_strike")).alias("wrong_call"))


In [179]:
# df_cum_ump_calls = (
#     data.group_by(["hp_official_id", "game_date", "game_pk"])
#     .agg(
#         n = pl.len(),
#         wrong=pl.col("wrong_call").sum().fill_null(0)
#     )
#     .sort(["hp_official_id", "game_date", "game_pk"])
#     .with_columns(
#         (pl.col("wrong").cum_sum().shift(1).over("hp_official_id") 
#         / pl.col("n").cum_sum().shift(1).over("hp_official_id")).alias("cum_ump_miss_rate").fill_null(0)
#     )
# )
# display(df_cum_ump_calls)

In [180]:
# data = data.join(
#     df_cum_ump_calls .select(["hp_official_id", "game_date", "game_pk", "cum_ump_miss_rate"]),
#     on=["hp_official_id", "game_date", "game_pk"],
#     how="left")

In [181]:
data = data.to_pandas()

In [182]:
pitch_mapping = {
    # Fastballs (Velocity & Movement variations)
    "FF": "Fastball",  # Four-Seam Fastball
    "SI": "Fastball",  # Sinker
    "FC": "Fastball",  # Cutter
    "FA": "Fastball",  # Other Fastball

    # Sliders (Horizontal movement)
    "SL": "Slider",    # TSlider
    "ST": "Slider",    # Sweeper
    "SV": "Slider",    # Slurve

    # Curveballs
    "CU": "Curveball", # Traditional Curveball
    "KC": "Curveball", # Knuckle-Curve
    "CS": "Curveball", # Slow Curve

    # Changeups
    "CH": "Changeup",  # Traditional Changeup
    "FS": "Changeup",  # Splitter (Finger-Split)
    "FO": "Changeup",  # Forkball
    "SC": "Changeup",  # Screwball (Fades opposite of a slider)

    # Junk
    "EP": "Junk",      # Eephus (Ultra-slow utility pitch)
    "KN": "Junk"       # Knuckleball
}

data["pitch_cat"] = data["pitch_type"].map(pitch_mapping)
data["count"] = data["balls"].astype(str) + "-" + data["strikes"].astype(str)
data["same_hand"] = (data["stand"] == data["p_throws"]).astype(int)
data["wrong_call"] = data["is_missed_ball"] | data["is_missed_strike"]
data['call'] = data['description']
data["month"] = pd.to_datetime(data["game_date"]).dt.month
data["zone"] = data["zone"].astype(int).astype(str) 
data["pitcher"] = data["pitcher"].astype(int).astype(str) 
data["batter"] = data["batter"].astype(int).astype(str) 
data["called_strike"] = (data["description"] == "called_strike").astype(int)

In [183]:
#data.info(verbose=True)

In [184]:
feature_cols = [
    "release_speed",
    "release_spin_rate",
    "pitch_cat",
    #"zone",
    "balls",
    "strikes",
    "inning",
    "outs_when_up",
    "inning_topbot",
    "stand",
    "p_throws",
    "n_thruorder_pitcher",
    "sz_top",
    "sz_bot",
    "called_strike",
    "pitcher",
    "batter",
    "pitch_number",
    "pitcher_days_since_prev_game",
    "attack_zone",
    "hp_official_id",
    "count",
    #"cum_ump_miss_rate"

]

response_cols = [
    "is_missed_ball",
    "is_missed_strike",
    "wrong_call"
    ]

In [185]:
# data.describe()

In [186]:
train_dir = Path("../data/processed/train")
train_dir.mkdir(exist_ok=True)

test_dir = Path("../data/processed/test")
test_dir.mkdir(exist_ok=True)

features_dir = Path("../data/processed/features")
features_dir.mkdir(parents=True,exist_ok=True)

response_cols_dir = Path("../data/processed/response_cols")
response_cols_dir.mkdir(exist_ok=True)

In [187]:
pd.Series(feature_cols, name="feature").to_csv(
    features_dir / "feature_cols.csv",
    index=False
)

pd.Series(response_cols, name="response").to_csv(
    response_cols_dir / "response_cols.csv",
    index=False
)

In [188]:
test_data = data[data["game_date"].dt.date > date(2025, 1, 1)]
train_data = data.drop(test_data.index)
test_data , train_data  = test_data.reset_index(drop=True), train_data.reset_index(drop=True)

In [189]:
train_path = train_dir / f"train_data.parquet"
train_data.to_parquet(train_path,index=False)

test_path = test_dir / f"test_data.parquet"
test_data.to_parquet(test_path,index=False)